# Chapter 29: Multi Session SLAM

<a href="../lite/lab/index.html?path=ch29_multisession_slam.ipynb" target="_blank" style="display:inline-block;padding:8px 16px;background:#1976d2;color:white;border-radius:4px;text-decoration:none;font-weight:bold">▶ Open in JupyterLite (editable, no install)</a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import least_squares

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

Monday: your robot maps the office. Tuesday: it maps it again. The
Monday map has errors in the kitchen. The Tuesday map has errors in the
conference room. Can you combine them so each map fixes the other's
weak spots?

**Multi session SLAM** merges multiple mapping sessions that share
overlapping areas. The shared regions act as anchors, letting errors
from one session be corrected by better data from another. This chapter
implements session merging from scratch: SVD alignment using shared
landmarks, cross session loop closures, and joint optimization.

```{admonition} What you will build
:class: tip

- Merge two SLAM sessions that share overlapping areas using shared landmark alignment
- Build a joint pose graph with cross session constraints
- Show that the merged map is more accurate than either session alone

**Real world application:** Robots that operate repeatedly in the same environment (delivery robots, warehouse AMRs) benefit enormously from combining maps across sessions.
```

```{admonition} Libraries and tools used in practice
:class: note

In this chapter we implement everything from scratch for learning. In production, engineers use these libraries:

| Library / Tool | What it does |
|---|---|
| **map_merge (ROS)** | Merges occupancy grid maps from multiple robots |
| **RTAB-Map** | Supports multi-session mapping with database persistence |

Implementing from scratch teaches you **why** these tools work. Using them in production saves you from reinventing the wheel.
```

## 29.1 Merging Trajectories: SVD Alignment

Two sessions explore overlapping areas. Session 1 covers rooms A + B.
Session 2 covers rooms B + C. The landmarks in room B appear in both
sessions, but in different coordinate frames.

To merge the sessions, we find the rigid transform (rotation $\mathbf{R}$
and translation $\mathbf{t}$) that best aligns the shared landmarks:

$$\min_{\mathbf{R}, \mathbf{t}} \sum_{i \in \text{shared}} \| \mathbf{R} \mathbf{p}_i^{(2)} + \mathbf{t} - \mathbf{p}_i^{(1)} \|^2$$

This has a closed form solution via SVD of the cross covariance matrix.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
np.random.seed(42)
n_poses_per_session = 15
sigma_odom = 0.15
sigma_obs = 0.2
# ─────────────────────────────────────────────────────────────────────────────

# Create a world with 3 rooms: A (left), B (center), C (right)
# Each room has landmarks
room_A_lm = np.array([[-8, 2], [-7, 5], [-9, 4], [-6, 1]])      # 4 landmarks
room_B_lm = np.array([[-1, 3], [0, 6], [1, 1], [-2, 5], [2, 4]]) # 5 shared
room_C_lm = np.array([[6, 2], [7, 5], [8, 3], [9, 1]])           # 4 landmarks

all_landmarks = np.vstack([room_A_lm, room_B_lm, room_C_lm])
n_A = len(room_A_lm)
n_B = len(room_B_lm)
n_C = len(room_C_lm)

# Session 1 trajectory: covers rooms A and B
t1 = np.linspace(0, np.pi, n_poses_per_session)
s1_gt = np.column_stack([-3 + 6*np.cos(t1), 3 + 2.5*np.sin(t1)])

# Session 2 trajectory: covers rooms B and C (in a DIFFERENT coordinate frame)
# Session 2 is rotated 15 degrees and translated
theta_true = np.radians(15)
R_true = np.array([[np.cos(theta_true), -np.sin(theta_true)],
                    [np.sin(theta_true),  np.cos(theta_true)]])
t_true = np.array([1.0, -0.5])

t2 = np.linspace(np.pi, 0, n_poses_per_session)
s2_gt_world = np.column_stack([4 + 6*np.cos(t2), 3 + 2.5*np.sin(t2)])

# Transform session 2 ground truth to its own frame
s2_gt_local = (R_true @ s2_gt_world.T).T + t_true

# Transform room B and C landmarks to session 2 frame
room_B_lm_s2 = (R_true @ room_B_lm.T).T + t_true
room_C_lm_s2 = (R_true @ room_C_lm.T).T + t_true

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.plot(s1_gt[:, 0], s1_gt[:, 1], 'steelblue', lw=2, marker='o', ms=4,
        label='Session 1 trajectory')
ax.scatter(room_A_lm[:, 0], room_A_lm[:, 1], c='steelblue', s=80,
           marker='^', zorder=5, label='Room A landmarks')
ax.scatter(room_B_lm[:, 0], room_B_lm[:, 1], c='forestgreen', s=100,
           marker='*', zorder=5, label='Room B (shared)')
ax.set_title('Session 1: Rooms A + B\n(world frame)', fontsize=12)
ax.set_aspect('equal'); ax.legend(fontsize=8); ax.set_xlim(-12, 12)

ax = axes[1]
ax.plot(s2_gt_local[:, 0], s2_gt_local[:, 1], 'tomato', lw=2, marker='s',
        ms=4, label='Session 2 trajectory')
ax.scatter(room_B_lm_s2[:, 0], room_B_lm_s2[:, 1], c='forestgreen', s=100,
           marker='*', zorder=5, label='Room B (shared)')
ax.scatter(room_C_lm_s2[:, 0], room_C_lm_s2[:, 1], c='tomato', s=80,
           marker='^', zorder=5, label='Room C landmarks')
ax.set_title('Session 2: Rooms B + C\n(session 2 frame)', fontsize=12)
ax.set_aspect('equal'); ax.legend(fontsize=8); ax.set_xlim(-4, 16)

ax = axes[2]
ax.plot(s1_gt[:, 0], s1_gt[:, 1], 'steelblue', lw=2, marker='o', ms=3,
        alpha=0.5, label='Session 1')
ax.plot(s2_gt_local[:, 0], s2_gt_local[:, 1], 'tomato', lw=2, marker='s',
        ms=3, alpha=0.5, label='Session 2 (own frame)')
ax.set_title('Overlay: sessions are misaligned!', fontsize=12)
ax.set_aspect('equal'); ax.legend(fontsize=9)

plt.suptitle('Two sessions in different coordinate frames', fontsize=14,
             fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
def svd_align(source, target):
    """
    Find R, t that minimizes sum || R * source[i] + t - target[i] ||^2.
    Uses SVD of the cross covariance matrix (Procrustes solution).
    
    Returns: R (2x2), t (2,), aligned_source (N x 2)
    """
    src_mean = source.mean(axis=0)
    tgt_mean = target.mean(axis=0)
    src_c = source - src_mean
    tgt_c = target - tgt_mean
    
    # Cross covariance
    H = src_c.T @ tgt_c
    U, S, Vt = np.linalg.svd(H)
    
    # Ensure proper rotation (det = +1)
    d = np.linalg.det(Vt.T @ U.T)
    D = np.diag([1, np.sign(d)])
    R = Vt.T @ D @ U.T
    
    t = tgt_mean - R @ src_mean
    aligned = (R @ source.T).T + t
    
    return R, t, aligned

print('SVD alignment function defined.')

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
alignment_noise = 0.15        # noise on shared landmark positions
# ─────────────────────────────────────────────────────────────────────────────

# Noisy shared landmark positions from each session
shared_s1 = room_B_lm + np.random.randn(n_B, 2) * alignment_noise
shared_s2 = room_B_lm_s2 + np.random.randn(n_B, 2) * alignment_noise

# Align session 2 to session 1 using shared landmarks
R_est, t_est, shared_aligned = svd_align(shared_s2, shared_s1)

# Transform all session 2 data to session 1 frame
s2_aligned = (R_est @ s2_gt_local.T).T + t_est
room_C_aligned = (R_est @ room_C_lm_s2.T).T + t_est
room_B_s2_aligned = (R_est @ room_B_lm_s2.T).T + t_est

# Compute alignment error
angle_est = np.degrees(np.arctan2(R_est[1, 0], R_est[0, 0]))
angle_true_inv = np.degrees(np.arctan2(-np.sin(theta_true), np.cos(theta_true)))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ax.scatter(shared_s1[:, 0], shared_s1[:, 1], c='steelblue', s=80,
           marker='^', label='Session 1 landmarks')
ax.scatter(shared_s2[:, 0], shared_s2[:, 1], c='tomato', s=80,
           marker='s', label='Session 2 landmarks (original)')
ax.scatter(shared_aligned[:, 0], shared_aligned[:, 1], c='orange', s=80,
           marker='D', label='Session 2 landmarks (aligned)')
for i in range(n_B):
    ax.plot([shared_s1[i, 0], shared_aligned[i, 0]],
            [shared_s1[i, 1], shared_aligned[i, 1]],
            'forestgreen', lw=1.5, alpha=0.7)
ax.set_aspect('equal'); ax.legend(fontsize=8)
ax.set_title('Shared landmarks: before and after alignment', fontsize=12)

ax = axes[1]
ax.plot(s1_gt[:, 0], s1_gt[:, 1], 'steelblue', lw=2, marker='o', ms=4,
        label='Session 1')
ax.plot(s2_aligned[:, 0], s2_aligned[:, 1], 'tomato', lw=2, marker='s',
        ms=4, label='Session 2 (aligned)')
ax.scatter(room_A_lm[:, 0], room_A_lm[:, 1], c='steelblue', s=60,
           marker='^', alpha=0.7)
ax.scatter(room_B_lm[:, 0], room_B_lm[:, 1], c='forestgreen', s=100,
           marker='*', zorder=5, label='Shared landmarks')
ax.scatter(room_C_aligned[:, 0], room_C_aligned[:, 1], c='tomato', s=60,
           marker='^', alpha=0.7)
ax.set_aspect('equal'); ax.legend(fontsize=9)
ax.set_title('Merged sessions: both in session 1 frame', fontsize=12)

plt.suptitle('SVD alignment merges two sessions', fontsize=14,
             fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

rmse = np.sqrt(np.mean(np.sum((shared_aligned - shared_s1)**2, axis=1)))
print(f'Alignment RMSE on shared landmarks: {rmse:.4f} m')
print(f'Estimated rotation: {angle_est:.2f} degrees')

**Observation:** The SVD alignment correctly recovers the rotation and
translation between the two coordinate frames. After alignment, both
sessions live in the same frame, and the map now covers all three rooms
(A, B, and C). The shared landmarks in room B serve as the "bridge"
connecting the two sessions.

## 29.2 Cross Session Loop Closures and Joint Optimization

After alignment, shared landmarks create **cross session constraints**.
These are analogous to loop closures, but between sessions rather than
within a single session. We build a joint pose graph with:

- Odometry edges within each session
- Observation edges from poses to landmarks within each session
- **Cross session edges** from shared landmark observations

Joint optimization produces a merged map better than either session alone.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
np.random.seed(55)
n_per_sess = 12
sigma_odom_j = 0.2
sigma_obs_j = 0.25
# ─────────────────────────────────────────────────────────────────────────────

# Ground truth landmarks for rooms A, B, C
lm_A = np.array([[-7, 3], [-6, 6], [-8, 5]])
lm_B = np.array([[-1, 4], [0, 7], [1, 2], [2, 5]])
lm_C = np.array([[6, 3], [7, 6], [8, 4]])
all_lm = np.vstack([lm_A, lm_B, lm_C])
nA, nB, nC = len(lm_A), len(lm_B), len(lm_C)
n_lm_total = nA + nB + nC

# Session 1 ground truth trajectory (rooms A + B)
t1g = np.linspace(0.2, np.pi - 0.2, n_per_sess)
s1g = np.column_stack([-3 + 5*np.cos(t1g), 4 + 2.5*np.sin(t1g)])

# Session 2 ground truth trajectory (rooms B + C)
t2g = np.linspace(np.pi - 0.2, 0.2, n_per_sess)
s2g = np.column_stack([4 + 5*np.cos(t2g), 4 + 2.5*np.sin(t2g)])

def generate_session_data(gt_poses, landmarks, sigma_odom, sigma_obs,
                          sensor_range=8.0):
    """Generate noisy odometry and observations for one session."""
    n = len(gt_poses)
    # Noisy odometry
    odom = np.diff(gt_poses, axis=0) + np.random.randn(n - 1, 2) * sigma_odom
    # Dead reckoning
    dr = np.zeros_like(gt_poses)
    dr[0] = gt_poses[0] + np.random.randn(2) * 0.1
    for i in range(n - 1):
        dr[i + 1] = dr[i] + odom[i]
    # Observations
    obs = []
    for i in range(n):
        for j in range(len(landmarks)):
            d = np.linalg.norm(gt_poses[i] - landmarks[j])
            if d < sensor_range:
                z = landmarks[j] - gt_poses[i] + np.random.randn(2) * sigma_obs
                obs.append((i, j, z))
    return odom, dr, obs

# Session 1 sees rooms A and B
s1_visible = np.vstack([lm_A, lm_B])
s1_lm_ids = list(range(nA)) + list(range(nA, nA + nB))
odom1, dr1, obs1_local = generate_session_data(s1g, s1_visible,
                                                sigma_odom_j, sigma_obs_j)
# Map local landmark indices to global
obs1 = [(pi, s1_lm_ids[lj], z) for pi, lj, z in obs1_local]

# Session 2 sees rooms B and C
s2_visible = np.vstack([lm_B, lm_C])
s2_lm_ids = list(range(nA, nA + nB)) + list(range(nA + nB, nA + nB + nC))
odom2, dr2, obs2_local = generate_session_data(s2g, s2_visible,
                                                sigma_odom_j, sigma_obs_j)
obs2 = [(pi, s2_lm_ids[lj], z) for pi, lj, z in obs2_local]

print(f'Session 1: {n_per_sess} poses, {len(obs1)} observations')
print(f'Session 2: {n_per_sess} poses, {len(obs2)} observations')
print(f'Shared landmarks (room B): {nB}')

In [ ]:
def joint_optimize(poses1_init, poses2_init, lm_init, obs1, obs2,
                   odom1, odom2, sigma_obs, sigma_odom):
    """Joint optimization of two sessions + shared landmarks."""
    n1 = len(poses1_init)
    n2 = len(poses2_init)
    n_lm = len(lm_init)
    
    def pack(p1, p2, lm):
        return np.concatenate([p1.ravel(), p2.ravel(), lm.ravel()])
    
    def unpack(x):
        p1 = x[:2*n1].reshape(n1, 2)
        p2 = x[2*n1:2*n1+2*n2].reshape(n2, 2)
        lm = x[2*n1+2*n2:].reshape(n_lm, 2)
        return p1, p2, lm
    
    def residuals(x):
        p1, p2, lm = unpack(x)
        res = []
        # Session 1 odometry
        for i in range(n1 - 1):
            r = (p1[i+1] - p1[i] - odom1[i]) / sigma_odom
            res.extend(r)
        # Session 2 odometry
        for i in range(n2 - 1):
            r = (p2[i+1] - p2[i] - odom2[i]) / sigma_odom
            res.extend(r)
        # Session 1 observations
        for (pi, lj, z) in obs1:
            r = (lm[lj] - p1[pi] - z) / sigma_obs
            res.extend(r)
        # Session 2 observations
        for (pi, lj, z) in obs2:
            r = (lm[lj] - p2[pi] - z) / sigma_obs
            res.extend(r)
        # Anchor session 1 first pose
        res.extend((p1[0] - poses1_init[0]) * 50.0)
        return np.array(res)
    
    x0 = pack(poses1_init, poses2_init, lm_init)
    result = least_squares(residuals, x0, method='lm',
                           max_nfev=5000 * len(x0))
    return unpack(result.x), result.cost

print('Joint optimizer defined.')

In [ ]:
def single_session_optimize(poses_init, lm_init, obs, odom,
                            sigma_obs, sigma_odom, lm_ids=None):
    """Optimize a single session with its own landmarks."""
    n_p = len(poses_init)
    if lm_ids is None:
        lm_ids = sorted(set(lj for _, lj, _ in obs))
    n_lm = len(lm_init)
    
    def pack(p, lm):
        return np.concatenate([p.ravel(), lm.ravel()])
    
    def unpack(x):
        p = x[:2*n_p].reshape(n_p, 2)
        lm = x[2*n_p:].reshape(n_lm, 2)
        return p, lm
    
    def residuals(x):
        p, lm = unpack(x)
        res = []
        for i in range(n_p - 1):
            r = (p[i+1] - p[i] - odom[i]) / sigma_odom
            res.extend(r)
        for (pi, lj, z) in obs:
            r = (lm[lj] - p[pi] - z) / sigma_obs
            res.extend(r)
        res.extend((p[0] - poses_init[0]) * 50.0)
        return np.array(res)
    
    x0 = pack(poses_init, lm_init)
    result = least_squares(residuals, x0, method='lm',
                           max_nfev=5000 * len(x0))
    return unpack(result.x)

# Initialize landmarks from observations
def init_landmarks(dr_poses, obs, n_lm):
    lm = np.zeros((n_lm, 2))
    ct = np.zeros(n_lm)
    for (pi, lj, z) in obs:
        lm[lj] += dr_poses[pi] + z
        ct[lj] += 1
    for j in range(n_lm):
        if ct[j] > 0:
            lm[j] /= ct[j]
        else:
            lm[j] = np.random.randn(2)
    return lm

# Session 1 alone
lm_init1 = init_landmarks(dr1, obs1, n_lm_total)
s1_opt, lm1_opt = single_session_optimize(dr1, lm_init1, obs1, odom1,
                                            sigma_obs_j, sigma_odom_j)

# Session 2 alone (need to align to world frame first for comparison)
lm_init2 = init_landmarks(dr2, obs2, n_lm_total)
s2_opt, lm2_opt = single_session_optimize(dr2, lm_init2, obs2, odom2,
                                            sigma_obs_j, sigma_odom_j)

# Joint optimization
lm_init_joint = np.zeros((n_lm_total, 2))
ct_j = np.zeros(n_lm_total)
for (pi, lj, z) in obs1:
    lm_init_joint[lj] += dr1[pi] + z
    ct_j[lj] += 1
for (pi, lj, z) in obs2:
    lm_init_joint[lj] += dr2[pi] + z
    ct_j[lj] += 1
for j in range(n_lm_total):
    if ct_j[j] > 0:
        lm_init_joint[j] /= ct_j[j]
    else:
        lm_init_joint[j] = all_lm[j] + np.random.randn(2) * 0.5

(sj1_opt, sj2_opt, lmj_opt), cost_j = joint_optimize(
    dr1, dr2, lm_init_joint, obs1, obs2, odom1, odom2,
    sigma_obs_j, sigma_odom_j)

print('All optimizations complete.')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Session 1 alone
ax = axes[0]
ax.plot(s1g[:, 0], s1g[:, 1], 'forestgreen', ls='--', lw=1.5, alpha=0.4,
        label='Truth')
ax.plot(s1_opt[:, 0], s1_opt[:, 1], 'steelblue', lw=2, marker='o', ms=4,
        label='Session 1 opt')
ax.scatter(all_lm[:, 0], all_lm[:, 1], c='forestgreen', s=50, marker='^',
           alpha=0.4)
vis_lm_mask1 = ct_j[:nA+nB] > 0  # landmarks session 1 can see
for j in range(nA + nB):
    if np.any(lm1_opt[j] != 0):
        ax.scatter(lm1_opt[j, 0], lm1_opt[j, 1], c='steelblue', s=60,
                   marker='^', zorder=5)
err1 = np.mean(np.linalg.norm(s1_opt - s1g, axis=1))
ax.set_title(f'Session 1 alone\npose error = {err1:.3f} m', fontsize=11)
ax.set_aspect('equal'); ax.legend(fontsize=8)

# Session 2 alone
ax = axes[1]
ax.plot(s2g[:, 0], s2g[:, 1], 'forestgreen', ls='--', lw=1.5, alpha=0.4,
        label='Truth')
ax.plot(s2_opt[:, 0], s2_opt[:, 1], 'tomato', lw=2, marker='s', ms=4,
        label='Session 2 opt')
ax.scatter(all_lm[:, 0], all_lm[:, 1], c='forestgreen', s=50, marker='^',
           alpha=0.4)
err2 = np.mean(np.linalg.norm(s2_opt - s2g, axis=1))
ax.set_title(f'Session 2 alone\npose error = {err2:.3f} m', fontsize=11)
ax.set_aspect('equal'); ax.legend(fontsize=8)

# Joint
ax = axes[2]
ax.plot(s1g[:, 0], s1g[:, 1], 'forestgreen', ls='--', lw=1.5, alpha=0.4)
ax.plot(s2g[:, 0], s2g[:, 1], 'forestgreen', ls='--', lw=1.5, alpha=0.4,
        label='Truth')
ax.plot(sj1_opt[:, 0], sj1_opt[:, 1], 'steelblue', lw=2, marker='o',
        ms=4, label='Session 1 (joint)')
ax.plot(sj2_opt[:, 0], sj2_opt[:, 1], 'tomato', lw=2, marker='s',
        ms=4, label='Session 2 (joint)')
ax.scatter(all_lm[:, 0], all_lm[:, 1], c='forestgreen', s=50, marker='^',
           alpha=0.4)
ax.scatter(lmj_opt[:, 0], lmj_opt[:, 1], c='orange', s=80, marker='*',
           zorder=5, label='Joint landmarks')
err_j1 = np.mean(np.linalg.norm(sj1_opt - s1g, axis=1))
err_j2 = np.mean(np.linalg.norm(sj2_opt - s2g, axis=1))
ax.set_title(f'Joint optimization\nS1 err = {err_j1:.3f}, S2 err = {err_j2:.3f} m',
             fontsize=11)
ax.set_aspect('equal'); ax.legend(fontsize=8)

plt.suptitle('Single session vs Joint optimization', fontsize=14,
             fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Error comparison bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pose errors
ax = axes[0]
labels = ['Session 1\nalone', 'Session 2\nalone', 'Session 1\njoint',
           'Session 2\njoint']
errors = [err1, err2, err_j1, err_j2]
colors = ['steelblue', 'tomato', 'steelblue', 'tomato']
alphas = [0.5, 0.5, 1.0, 1.0]
bars = ax.bar(labels, errors, color=colors, alpha=alphas)
for b, v in zip(bars, errors):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.005,
            f'{v:.3f}', ha='center', fontsize=10)
ax.set_ylabel('Mean pose error (m)', fontsize=12)
ax.set_title('Pose accuracy: alone vs joint', fontsize=12)

# Landmark errors for shared landmarks (room B)
ax = axes[1]
lm_err_s1 = np.mean(np.linalg.norm(lm1_opt[nA:nA+nB] - lm_B, axis=1))
lm_err_s2 = np.mean(np.linalg.norm(lm2_opt[nA:nA+nB] - lm_B, axis=1))
lm_err_joint = np.mean(np.linalg.norm(lmj_opt[nA:nA+nB] - lm_B, axis=1))

labels2 = ['Session 1\nalone', 'Session 2\nalone', 'Joint']
errs2 = [lm_err_s1, lm_err_s2, lm_err_joint]
cols2 = ['steelblue', 'tomato', 'orange']
bars2 = ax.bar(labels2, errs2, color=cols2, alpha=0.8)
for b, v in zip(bars2, errs2):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.005,
            f'{v:.3f}', ha='center', fontsize=10)
ax.set_ylabel('Mean landmark error (m)', fontsize=12)
ax.set_title('Shared landmark accuracy (Room B)', fontsize=12)

plt.tight_layout(); plt.show()
print('Joint optimization improves both sessions because the shared')
print('landmarks provide cross session constraints.')

In [ ]:
# Visualize cross session edges
fig, ax = plt.subplots(figsize=(10, 7))

ax.plot(sj1_opt[:, 0], sj1_opt[:, 1], 'steelblue', lw=2, marker='o',
        ms=5, label='Session 1', zorder=3)
ax.plot(sj2_opt[:, 0], sj2_opt[:, 1], 'tomato', lw=2, marker='s',
        ms=5, label='Session 2', zorder=3)
ax.scatter(lmj_opt[:, 0], lmj_opt[:, 1], c='orange', s=100, marker='*',
           zorder=5, label='Landmarks')

# Draw observation edges from Session 1 to shared landmarks
for (pi, lj, z) in obs1:
    if nA <= lj < nA + nB:  # shared landmark
        ax.plot([sj1_opt[pi, 0], lmj_opt[lj, 0]],
                [sj1_opt[pi, 1], lmj_opt[lj, 1]],
                'steelblue', alpha=0.2, lw=0.8)

# Draw observation edges from Session 2 to shared landmarks
for (pi, lj, z) in obs2:
    if nA <= lj < nA + nB:  # shared landmark
        ax.plot([sj2_opt[pi, 0], lmj_opt[lj, 0]],
                [sj2_opt[pi, 1], lmj_opt[lj, 1]],
                'tomato', alpha=0.2, lw=0.8)

ax.set_aspect('equal'); ax.legend(fontsize=10)
ax.set_title('Cross session edges through shared landmarks',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print('Shared landmarks act as bridges between sessions.')
print('Both sessions constrain the landmark positions, and in return')
print('the landmarks constrain both sessions.')

---

## Capstone: Three Sessions with Pairwise Overlaps

Three mapping sessions:
- **Session A:** Covers zone 1 + zone 2
- **Session B:** Covers zone 2 + zone 3
- **Session C:** Covers zone 3 + zone 1

Every pair shares one zone. We merge them progressively and show the
map improving at each step:
1. A alone
2. A + B merged
3. A + B + C merged

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
np.random.seed(88)
n_per_cap = 10
sigma_odom_c = 0.2
sigma_obs_c = 0.2
# ─────────────────────────────────────────────────────────────────────────────

# Three zones arranged in a triangle
zone1 = np.array([[-5, 0], [-4, 2], [-3, -1]])     # top left
zone2 = np.array([[3, 0], [4, 2], [5, -1]])         # top right
zone3 = np.array([[-1, -5], [0, -3], [1, -6]])      # bottom
all_zones = np.vstack([zone1, zone2, zone3])
nz1, nz2, nz3 = len(zone1), len(zone2), len(zone3)
n_lm_cap = nz1 + nz2 + nz3

# Session A covers zones 1 + 2 (top arc)
ta = np.linspace(0.8*np.pi, 0.2*np.pi, n_per_cap)
sa_gt = np.column_stack([6*np.cos(ta), 6*np.sin(ta) - 3])

# Session B covers zones 2 + 3 (right arc)
tb = np.linspace(-0.2*np.pi, -0.8*np.pi, n_per_cap)
sb_gt = np.column_stack([6*np.cos(tb) + 1, 6*np.sin(tb)])

# Session C covers zones 3 + 1 (left arc)
tc = np.linspace(1.3*np.pi, 0.7*np.pi, n_per_cap)
sc_gt = np.column_stack([6*np.cos(tc) - 1, 6*np.sin(tc)])

# Session A sees zones 1 + 2
sa_vis = np.vstack([zone1, zone2])
sa_ids = list(range(nz1)) + list(range(nz1, nz1 + nz2))
od_a, dr_a, obs_a_local = generate_session_data(sa_gt, sa_vis,
                                                  sigma_odom_c, sigma_obs_c, 10)
obs_a = [(pi, sa_ids[lj], z) for pi, lj, z in obs_a_local]

# Session B sees zones 2 + 3
sb_vis = np.vstack([zone2, zone3])
sb_ids = list(range(nz1, nz1 + nz2)) + list(range(nz1 + nz2, n_lm_cap))
od_b, dr_b, obs_b_local = generate_session_data(sb_gt, sb_vis,
                                                  sigma_odom_c, sigma_obs_c, 10)
obs_b = [(pi, sb_ids[lj], z) for pi, lj, z in obs_b_local]

# Session C sees zones 3 + 1
sc_vis = np.vstack([zone3, zone1])
sc_ids = list(range(nz1 + nz2, n_lm_cap)) + list(range(nz1))
od_c, dr_c, obs_c_local = generate_session_data(sc_gt, sc_vis,
                                                  sigma_odom_c, sigma_obs_c, 10)
obs_c = [(pi, sc_ids[lj], z) for pi, lj, z in obs_c_local]

print(f'Session A: {len(obs_a)} obs (zones 1+2)')
print(f'Session B: {len(obs_b)} obs (zones 2+3)')
print(f'Session C: {len(obs_c)} obs (zones 3+1)')

In [ ]:
def multi_session_optimize(dr_list, obs_list, odom_list, n_lm, sigma_obs,
                           sigma_odom):
    """Optimize multiple sessions jointly."""
    n_sessions = len(dr_list)
    n_poses_list = [len(dr) for dr in dr_list]
    total_poses = sum(n_poses_list)
    
    # Flatten: [poses_s1, poses_s2, ..., landmarks]
    dim = 2 * total_poses + 2 * n_lm
    
    def pack(poses_list, lm):
        parts = [p.ravel() for p in poses_list] + [lm.ravel()]
        return np.concatenate(parts)
    
    def unpack(x):
        poses = []
        idx = 0
        for n in n_poses_list:
            poses.append(x[idx:idx+2*n].reshape(n, 2))
            idx += 2*n
        lm = x[idx:].reshape(n_lm, 2)
        return poses, lm
    
    def residuals(x):
        poses, lm = unpack(x)
        res = []
        for s in range(n_sessions):
            p = poses[s]
            od = odom_list[s]
            ob = obs_list[s]
            # Odometry
            for i in range(len(p) - 1):
                r = (p[i+1] - p[i] - od[i]) / sigma_odom
                res.extend(r)
            # Observations
            for (pi, lj, z) in ob:
                r = (lm[lj] - p[pi] - z) / sigma_obs
                res.extend(r)
            # Anchor first pose
            res.extend((p[0] - dr_list[s][0]) * 50.0)
        return np.array(res)
    
    # Initialize landmarks
    lm_init = np.zeros((n_lm, 2))
    ct = np.zeros(n_lm)
    for s in range(n_sessions):
        for (pi, lj, z) in obs_list[s]:
            lm_init[lj] += dr_list[s][pi] + z
            ct[lj] += 1
    for j in range(n_lm):
        if ct[j] > 0:
            lm_init[j] /= ct[j]
        else:
            lm_init[j] = np.random.randn(2)
    
    x0 = pack(dr_list, lm_init)
    result = least_squares(residuals, x0, method='lm',
                           max_nfev=5000 * len(x0))
    return unpack(result.x)

# Stage 1: A alone
lm_a_init = init_landmarks(dr_a, obs_a, n_lm_cap)
pa_only, lma_only = single_session_optimize(dr_a, lm_a_init, obs_a,
                                             od_a, sigma_obs_c, sigma_odom_c)

# Stage 2: A + B
ab_poses, ab_lm = multi_session_optimize(
    [dr_a, dr_b], [obs_a, obs_b], [od_a, od_b], n_lm_cap,
    sigma_obs_c, sigma_odom_c)

# Stage 3: A + B + C
abc_poses, abc_lm = multi_session_optimize(
    [dr_a, dr_b, dr_c], [obs_a, obs_b, obs_c], [od_a, od_b, od_c],
    n_lm_cap, sigma_obs_c, sigma_odom_c)

print('All three stages complete.')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
gt_sessions = [sa_gt, sb_gt, sc_gt]
sess_colors = ['steelblue', 'tomato', 'orange']
sess_labels = ['Session A', 'Session B', 'Session C']

# Stage 1: A alone
ax = axes[0]
ax.plot(sa_gt[:, 0], sa_gt[:, 1], 'forestgreen', ls='--', lw=1.5, alpha=0.4)
ax.plot(pa_only[:, 0], pa_only[:, 1], 'steelblue', lw=2, marker='o', ms=4)
ax.scatter(all_zones[:, 0], all_zones[:, 1], c='forestgreen', s=50,
           marker='^', alpha=0.4)
err_a = np.mean(np.linalg.norm(pa_only - sa_gt, axis=1))
ax.set_title(f'Stage 1: A alone\nerror = {err_a:.3f} m', fontsize=11)
ax.set_aspect('equal')

# Stage 2: A + B
ax = axes[1]
for gts in gt_sessions[:2]:
    ax.plot(gts[:, 0], gts[:, 1], 'forestgreen', ls='--', lw=1.5, alpha=0.3)
for s, col, lab in zip(ab_poses, sess_colors[:2], sess_labels[:2]):
    ax.plot(s[:, 0], s[:, 1], col, lw=2, marker='o', ms=3, label=lab)
ax.scatter(all_zones[:, 0], all_zones[:, 1], c='forestgreen', s=50,
           marker='^', alpha=0.4)
err_ab = np.mean([np.mean(np.linalg.norm(ab_poses[i] - gt_sessions[i], axis=1))
                  for i in range(2)])
ax.set_title(f'Stage 2: A + B\nmean error = {err_ab:.3f} m', fontsize=11)
ax.set_aspect('equal'); ax.legend(fontsize=8)

# Stage 3: A + B + C
ax = axes[2]
for gts in gt_sessions:
    ax.plot(gts[:, 0], gts[:, 1], 'forestgreen', ls='--', lw=1.5, alpha=0.3)
for s, col, lab in zip(abc_poses, sess_colors, sess_labels):
    ax.plot(s[:, 0], s[:, 1], col, lw=2, marker='o', ms=3, label=lab)
ax.scatter(abc_lm[:, 0], abc_lm[:, 1], c='orange', s=80, marker='*',
           zorder=5, label='Joint landmarks')
ax.scatter(all_zones[:, 0], all_zones[:, 1], c='forestgreen', s=50,
           marker='^', alpha=0.4)
err_abc = np.mean([np.mean(np.linalg.norm(abc_poses[i] - gt_sessions[i], axis=1))
                   for i in range(3)])
ax.set_title(f'Stage 3: A + B + C\nmean error = {err_abc:.3f} m', fontsize=11)
ax.set_aspect('equal'); ax.legend(fontsize=8)

plt.suptitle('Progressive session merging: more sessions = better map',
             fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout(); plt.show()

In [ ]:
# Convergence bar chart
fig, ax = plt.subplots(figsize=(8, 5))
stages = ['A alone', 'A + B', 'A + B + C']
stage_errors = [err_a, err_ab, err_abc]
bars = ax.bar(stages, stage_errors, color=['steelblue', 'orange', 'forestgreen'],
              alpha=0.8)
for b, v in zip(bars, stage_errors):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.005,
            f'{v:.3f}', ha='center', fontsize=12, fontweight='bold')
ax.set_ylabel('Mean pose error (m)', fontsize=12)
ax.set_title('Adding sessions progressively improves accuracy', fontsize=13)
plt.tight_layout(); plt.show()

improvement = (1 - err_abc / err_a) * 100
print(f'Improvement from A alone to A+B+C: {improvement:.1f}%')

In [ ]:
# Per-landmark error comparison
lm_err_a_only = np.linalg.norm(lma_only - all_zones, axis=1)
lm_err_abc = np.linalg.norm(abc_lm - all_zones, axis=1)

fig, ax = plt.subplots(figsize=(10, 5))
x_pos = np.arange(n_lm_cap)
w = 0.35

# Only show landmarks that A can see
ax.bar(x_pos - w/2, lm_err_a_only, w, color='steelblue', alpha=0.7,
       label='A alone')
ax.bar(x_pos + w/2, lm_err_abc, w, color='orange', alpha=0.7,
       label='A + B + C')

# Label zones
ax.axvspan(-0.5, nz1 - 0.5, alpha=0.08, color='steelblue')
ax.axvspan(nz1 - 0.5, nz1 + nz2 - 0.5, alpha=0.08, color='tomato')
ax.axvspan(nz1 + nz2 - 0.5, n_lm_cap - 0.5, alpha=0.08, color='orange')
ax.text(nz1/2 - 0.5, ax.get_ylim()[1]*0.9, 'Zone 1', ha='center', fontsize=10)
ax.text(nz1 + nz2/2 - 0.5, ax.get_ylim()[1]*0.9, 'Zone 2', ha='center', fontsize=10)
ax.text(nz1 + nz2 + nz3/2 - 0.5, ax.get_ylim()[1]*0.9, 'Zone 3', ha='center', fontsize=10)

ax.set_xlabel('Landmark index', fontsize=12)
ax.set_ylabel('Position error (m)', fontsize=12)
ax.set_title('Per landmark error: single vs multi session', fontsize=13)
ax.set_xticks(x_pos)
ax.legend(fontsize=10)
plt.tight_layout(); plt.show()

print('Multi session SLAM improves landmarks in ALL zones,')
print('because cross session constraints propagate globally.')

**Capstone observations:**

- Each additional session brings new constraints that improve the
  entire map, not just the newly covered area.
- The shared landmarks create "bridges" between sessions. The more
  bridges, the stronger the connection.
- With three pairwise overlapping sessions forming a cycle (A with B,
  B with C, C with A), the constraint structure is similar to a loop
  closure, providing maximum mutual correction.
- In practice, the challenge is **identifying** which landmarks are
  shared across sessions (place recognition).

---

## Exercises

### Exercise 29.1: Minimum Shared Landmarks

How many shared landmarks are needed for reliable alignment? Sweep the
number of shared landmarks from 2 to 10 and plot alignment error vs.
count. What is the minimum for sub-meter accuracy?

In [ ]:
# Your code here

### Exercise 29.2: Noisy Correspondences

What happens when shared landmark identities are partially wrong?
Introduce 1, 2, or 3 wrong correspondences (landmark A in session 1
matched to the wrong landmark in session 2). Show how alignment
degrades. Can you detect the wrong correspondences from residuals?

In [ ]:
# Your code here

### Exercise 29.3: Temporal Separation

Simulate two sessions separated by "time" where the environment changes:
add random noise (0.0 to 0.5 m) to shared landmark positions between
sessions. Plot merged map error vs. landmark drift. At what drift level
does merging hurt rather than help?

In [ ]:
# Your code here

### Exercise 29.4: Session Weighting (challenge)

If one session has higher quality sensors (lower noise), should its
data be weighted more heavily? Simulate sessions with different noise
levels ($\sigma_1 = 0.1$ m, $\sigma_2 = 0.5$ m). Compare joint
optimization with equal weights vs. proper information weighting.

In [ ]:
# Your code here